In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv(r"C:\Telco\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")   # change to your filename
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()
print()
print("Missing values:")
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Unique customerIDs:", df['customerID'].nunique(), "of", len(df))

Duplicate rows: 0
Unique customerIDs: 7043 of 7043


In [5]:
print("TotalCharges dtype:", df['TotalCharges'].dtype)

# Rows where it is blank or not convertible to a number
bad = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isnull()]
print("Non-numeric TotalCharges rows:", len(bad))
bad[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

TotalCharges dtype: str
Non-numeric TotalCharges rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [6]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3))
print()

for col in df.select_dtypes('object').columns:
    if col in ['customerID', 'TotalCharges']:
        continue
    print(col, "->", df[col].unique().tolist())

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64

gender -> ['Female', 'Male']
Partner -> ['Yes', 'No']
Dependents -> ['No', 'Yes']
PhoneService -> ['No', 'Yes']
MultipleLines -> ['No phone service', 'No', 'Yes']
InternetService -> ['DSL', 'Fiber optic', 'No']
OnlineSecurity -> ['No', 'Yes', 'No internet service']
OnlineBackup -> ['Yes', 'No', 'No internet service']
DeviceProtection -> ['No', 'Yes', 'No internet service']
TechSupport -> ['No', 'Yes', 'No internet service']
StreamingTV -> ['No', 'Yes', 'No internet service']
StreamingMovies -> ['No', 'Yes', 'No internet service']
Contract -> ['Month-to-month', 'One year', 'Two year']
PaperlessBilling -> ['Yes', 'No']
PaymentMethod -> ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
Churn -> ['No', 'Yes']


C:\Users\deepa\AppData\Local\Temp\ipykernel_24492\1675355324.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes('object').columns:


In [7]:
data = df.copy()

The blank entries become NaN. In the Telco data these are customers with tenure 0 who haven't been billed yet.

In [8]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
print("Missing TotalCharges:", data['TotalCharges'].isnull().sum())

Missing TotalCharges: 11


In [9]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
data[num_cols].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7032.00
mean,32.37,64.76,2283.30
std,24.56,30.09,2266.77
min,0.00,18.25,18.80
25%,9.00,35.50,401.45
50%,29.00,70.35,1397.48
75%,55.00,89.85,3794.74
max,72.00,118.75,8684.80


In [10]:
# New customers with tenure 0 have not been billed, so 0 is the logical value
mask = data['TotalCharges'].isnull()
print("Tenure of these rows:", data.loc[mask, 'tenure'].unique())

data.loc[mask, 'TotalCharges'] = 0
print("Missing TotalCharges now:", data['TotalCharges'].isnull().sum())

Tenure of these rows: [0]
Missing TotalCharges now: 0


Encode churn with 0/1

In [11]:
if data['Churn'].dtype == 'object':
    data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

print(data['Churn'].value_counts())

Churn
No     5174
Yes    1869
Name: count, dtype: int64


These values carry the same information as InternetService = No and PhoneService = No, which are already in their own columns. Collapsing them to "No" removes the duplication.

In [12]:
internet_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in internet_cols:
    data[col] = data[col].replace('No internet service', 'No')

data['MultipleLines'] = data['MultipleLines'].replace('No phone service', 'No')

for col in internet_cols + ['MultipleLines']:
    print(col, "->", data[col].unique().tolist())

OnlineSecurity -> ['No', 'Yes']
OnlineBackup -> ['Yes', 'No']
DeviceProtection -> ['No', 'Yes']
TechSupport -> ['No', 'Yes']
StreamingTV -> ['No', 'Yes']
StreamingMovies -> ['No', 'Yes']
MultipleLines -> ['No', 'Yes']


In [13]:
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines'] + internet_cols

for col in yes_no_cols:
    data[col] = data[col].map({'Yes': 1, 'No': 0})

data[yes_no_cols].head()

,Partner,Dependents,PhoneService,PaperlessBilling,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,1,0,0,1,0,0,1,0,0,0,0
1,0,0,1,0,0,1,0,1,0,0,0
2,0,0,1,1,0,1,1,0,0,0,0
3,0,0,0,0,0,1,0,1,1,0,0
4,0,0,1,1,0,0,0,0,0,0,0


In [14]:
data = data.drop(columns=['customerID'])

In [15]:
print(data.shape)
print()
print("Missing values:", data.isnull().sum().sum())
print()
data.dtypes

(7043, 20)

Missing values: 0



gender                  str
SeniorCitizen         int64
Partner               int64
Dependents            int64
tenure                int64
PhoneService          int64
MultipleLines         int64
InternetService         str
OnlineSecurity        int64
OnlineBackup          int64
DeviceProtection      int64
TechSupport           int64
StreamingTV           int64
StreamingMovies       int64
Contract                str
PaperlessBilling      int64
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object

In [17]:
import pandas as pd

# 1. Convert blanks to NaN, then fill with 0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("NaNs before fill:", df["TotalCharges"].isna().sum())            # 11
print(df.loc[df["TotalCharges"].isna(), "tenure"].unique())            # should be [0]

df["TotalCharges"] = df["TotalCharges"].fillna(0)

# 2. Drop the ID column (not useful for modeling)
df = df.drop(columns=["customerID"], errors="ignore")

# 3. Verify
print(df["TotalCharges"].dtype)          # float64
print(df.isna().sum().sum())             # 0
print(df.dtypes)

# 4. Save
df.to_csv("telco_clean.csv", index=False)

NaNs before fill: 11
[0]
float64
0
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


In [18]:
df.to_csv("telco_clean.csv", index=False)